In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import psycopg2
from bs4 import BeautifulSoup
from collections import defaultdict, OrderedDict
import skimpy
import numpy as np
import math
import os
os.environ["PYTHONUTF8"] = "1"
os.environ["PGPASSFILE"] = ""


In [4]:

url_wiki="https://fr.wikipedia.org/wiki/Liste_des_départements_français_classés_par_population_et_superficie"

In [5]:
content_wiki=requests.get(url_wiki,headers = {
    'User-Agent': 'MonRobotFoot/1.0 (contact: monadresse@email.com) Python-requests'})
soup=BeautifulSoup(content_wiki.text)

In [6]:
table=soup.find("table",class_="wikitable")
sup_list=table.find_all("sup",class_="reference")
for sup in sup_list:
     sup.decompose()
#print("table: "+str(table))
tbody=table.find("tbody")
#print("tbody: "+str(tbody))
tr_list=tbody.find_all("tr")
list_rows=[]

for tr in tr_list:
    if tr.find("th")!= None:
         print("continue")
         continue
    #print("tr: "+str(tr))
    td_list=tr.find_all("td")
    #print("td_list:"+str(td_list))
    row=[]
    for num_col,td in enumerate(td_list):
        #print("td: "+str(td))
        if td.has_attr("colspan"):
            #print(td)
            colspan=td["colspan"]
            row.extend([None]*(int(colspan)-1))
        row.append(td.text.strip())
    if len(row)!=14:
            raise ValueError(f"Wrong col length: {len(row)} Values: {row}")    
    list_rows.append(row)
table_df=pd.DataFrame(list_rows)


continue
continue


In [7]:

table_df=table_df.rename(columns={1:"Numero_departement",2:"Nom",11:"Population"})
table_df=table_df[["Numero_departement","Nom","Population"]]
print(table_df.head(100))

   Numero_departement                                      Nom Population
0                  59                                     Nord  2 615 635
1                  75                                    Paris  2 103 778
2                  13                         Bouches-du-Rhône  2 087 658
3                 6AE         Collectivité européenne d'Alsace  1 934 548
4                  69  Circonscription départementale du Rhône  1 914 667
..                ...                                      ...        ...
95                 52                              Haute-Marne    168 331
96                 2A                             Corse-du-Sud    168 306
97                 04                  Alpes-de-Haute-Provence    168 054
98                 09                                   Ariège    155 722
99                 15                                   Cantal    144 196

[100 rows x 3 columns]


In [8]:
table_df["Population"]=table_df["Population"].str.replace("\xa0", "", regex=False)
print(table_df["Population"])

0      2615635
1      2103778
2      2087658
3      1934548
4      1914667
        ...   
99      144196
100     143467
101     140255
102     115527
103      76486
Name: Population, Length: 104, dtype: str


In [9]:
print(table_df.to_string())

    Numero_departement                                      Nom Population
0                   59                                     Nord    2615635
1                   75                                    Paris    2103778
2                   13                         Bouches-du-Rhône    2087658
3                  6AE         Collectivité européenne d'Alsace    1934548
4                   69  Circonscription départementale du Rhône    1914667
5                   93                        Seine-Saint-Denis    1704316
6                   33                                  Gironde    1690493
7                   92                           Hauts-de-Seine    1654712
8                   44                         Loire-Atlantique    1487570
9                   78                                 Yvelines    1485086
10                  31                            Haute-Garonne    1471468
11                  77                           Seine-et-Marne    1468108
12                  62   

In [10]:
table_df["Population"]=table_df["Population"].astype(int)

In [11]:

print(os.environ.get("APPDATA"))
print(os.environ.get("USERPROFILE"))

C:\Users\Utilisateur\AppData\Roaming
C:\Users\Utilisateur


In [2]:

conn = psycopg2.connect(host="localhost", user="postgres", password="jaune2000", port=5432)

In [3]:
conn=psycopg2.connect(user="postgres",host="localhost",password="jaune2000", port=5432,database="sandbox_db")
cur=conn.cursor()

In [ ]:
def create_tables(cur):
    cur.execute("DROP TABLE IF EXISTS bornes")
    cur.execute("DROP TABLE IF EXISTS departements CASCADE")

    query="""CREATE TABLE departements(
    code VARCHAR(5) PRIMARY KEY,
    nom VARCHAR(255),
    population INTEGER
    )"""
    cur.execute(query)


    query="""CREATE TABLE bornes(
    id BIGINT GENERATED ALWAYS AS IDENTITY, 
    nom_operateur VARCHAR(255),
    code_insee_commune VARCHAR(5),
    puissance_nominale FLOAT,
    code_departement VARCHAR(5) REFERENCES departements(code)

    )"""
    cur.execute(query)




In [15]:
create_tables(cur)

In [16]:
data_departements_to_insert=table_df.values.tolist()
cur.executemany("INSERT INTO departements VALUES (%s,%s,%s)",data_departements_to_insert)

In [17]:
cur.execute("SELECT * FROM departements LIMIT 5")
cur.fetchall()

[('59', 'Nord', 2615635),
 ('75', 'Paris', 2103778),
 ('13', 'Bouches-du-Rhône', 2087658),
 ('6AE', "Collectivité européenne d'Alsace", 1934548),
 ('69', 'Circonscription départementale du Rhône', 1914667)]

In [18]:

url_api="https://www.data.gouv.fr/api/1/datasets/5448d3e0c751df01f85d0572/"

api_json=requests.get(url_api).json()



In [19]:
url_data=api_json["resources"][0]["url"]

In [20]:
data_df=pd.read_csv(StringIO(requests.get(url_data).text))

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_21236\2529255909.py:1: DtypeWarning: Columns (0: code_insee_commune, 1: prise_type_ef, 2: prise_type_2, 3: prise_type_combo_ccs, 4: prise_type_chademo, 5: prise_type_autre, 6: paiement_acte, 7: paiement_cb, 8: reservation, 9: station_deux_roues) have mixed types. Specify dtype option on import or set low_memory=False.
  data_df=pd.read_csv(StringIO(requests.get(url_data).text))


In [21]:
data_bornes_df=data_df[["nom_operateur","code_insee_commune","puissance_nominale"]].copy()

In [22]:
cur.execute("SELECT * FROM bornes LIMIT 5")
cur.fetchall()

[]

In [23]:
def insert_into_tables(cur,data_bornes_df):
    data_borns_to_insert=data_bornes_df.values.tolist()
    cur.executemany("INSERT INTO bornes VALUES (DEFAULT,%s,%s,%s)",data_borns_to_insert)

In [24]:
def soft_float_to_int(serie):
        new_list=[]
        for value in serie:
            if isinstance(value,float):  #convert float to int if no number after decimal
                if value % 1 == 0:
                    value=int(value)
            new_list.append(value)        
        return pd.Series(new_list)

In [27]:
data_bornes_df["code_insee_commune"]

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
          ...  
228491    59359
228492      NaN
228493      NaN
228494      NaN
228495      NaN
Name: code_insee_commune, Length: 228496, dtype: object

In [31]:
data_bornes_df["code_insee_commune"]=soft_float_to_int(data_bornes_df["code_insee_commune"].astype(object))

In [32]:
data_bornes_df["code_insee_commune"]=data_bornes_df["code_insee_commune"].astype(str)

In [33]:
data_bornes_df[data_bornes_df["code_insee_commune"].isna()].head(3)



,nom_operateur,code_insee_commune,puissance_nominale
0,ChargePoint,NaN,22.0
1,ChargePoint,NaN,22.0
2,ChargePoint,NaN,11.0


In [35]:
def get_proportion_na(df):
    print("Valeurs manquantes: \n")  #comtpe valeurs manquantes
    for col in df.columns:
        nb_na=df[col].isna().sum()
        print(str(col)+" : "+str(nb_na)+" ("+str(nb_na/len(df)*100)+"%")

In [36]:
get_proportion_na(data_bornes_df)

Valeurs manquantes: 

nom_operateur : 3985 (1.7440130242980183%
code_insee_commune : 66838 (29.251277921714163%
puissance_nominale : 0 (0.0%


In [37]:
for value in data_bornes_df["puissance_nominale"].unique():
    print(value)

22.0
11.0
44.0
36.0
14.8
200.0
50.0
300.0
22.08
150.0
60.0
100.0
400.0
240.0
560.0
3.4
7.4
3.7
24.0
30.0
7.3
120.0
75.0
7.0
360.0
180.0
7.36
149.999
22.078
30.4
2.3
11.04
3.68
90.0
0.0
1.84
38.4
21.888
2.88
14.72
3.0
6.0
4.6
2.2
5.5
43.0
160.0
350.0
250.0
14.0
600.0
280.0
29.0
307.0
150.4
29.6
40.0
25.0
20.0
18.0
62.5
66.0
320.0
47.0
500.0
32.0
375.0
132.0
352.0
2.0
72.0
144.0
116.0
108.0
156.0
88.0
232.0
192.0
225.0
148.0
94.0
110.0
164.0
326.0
172.0
52.0
678.0
1242.0
258.0
630.0
82.0
1500.0
372.0
900.0
170.0
118.0
38.0
7.328
210.0
62.0
184.0
63.0
23.0
26.0
27.0
130.0
80.0
3.5
42.0
158.0
211.0
113.0
239.0
396.0
105.0
176.0
53.0
91.0
101.0
12.0
1.6
220.0
2.299999952316284
7.400000095367432
16.0
104.0
204.0
9.0
114.0
64.0
21.0
19.0
444.0
44.4
124.0
175.0
165.0
1.8
14.49
3.22
20.01
3.681
4.0
49.999
43.47
6.9
149.997
119.1
54.0
480.0
1601.6
319.6
163.2
159.8
450.0
125.0
748.0
58.0
13.0
23.04
65.0
22.1
22.2
178.0
187.5
111.0
187.0
22.17025
146.0
10.0
70.0
181.0
39.0
86.0
0.0001
99.0
4.8
12

In [38]:
data_bornes_df.head(5)

,nom_operateur,code_insee_commune,puissance_nominale
0,ChargePoint,NaN,22.0
1,ChargePoint,NaN,22.0
2,ChargePoint,NaN,11.0
3,ChargePoint,NaN,11.0
4,ChargePoint,NaN,22.0


In [39]:
list_code_departement=[]
cur.execute("SELECT code FROM departements")
for row in cur.fetchall():
    list_code_departement.append(row[0])

code_commune_to_code_departement_list=[]
for code_commune in data_bornes_df["code_insee_commune"]:
    if not isinstance(code_commune,str):
        if math.isnan(code_commune):
            code_commune_to_code_departement_list.append(None)  
    elif code_commune[0:3] in list_code_departement:
        code_commune_to_code_departement_list.append(code_commune[0:3])
    elif code_commune[0:2] in list_code_departement:
        code_commune_to_code_departement_list.append(code_commune[0:2]) 
    else:
        code_commune_to_code_departement_list.append(None)


In [40]:
data_bornes_df["departement"]=code_commune_to_code_departement_list
data_bornes_df["departement"]=data_bornes_df["departement"].replace({np.nan: None})
data_bornes_df=data_bornes_df.replace({np.nan: None})
print(data_bornes_df["departement"].head(5))

0    None
1    None
2    None
3    None
4    None
Name: departement, dtype: object


In [ ]:
data_bornes_df.values.tolist()[0:5]

[['ChargePoint', None, 22.0, None],
 ['ChargePoint', None, 22.0, None],
 ['ChargePoint', None, 11.0, None],
 ['ChargePoint', None, 11.0, None],
 ['ChargePoint', None, 22.0, None]]

In [ ]:
"""for row in data_bornes_df.values.tolist():
    try:
        cur.execute("INSERT INTO bornes VALUES (DEFAULT,%s,%s,%s,%s)", row)
    except Exception as e:
        print("Failed row:", row)
        raise
"""

'for row in data_bornes_df.values.tolist():\n    try:\n        cur.execute("INSERT INTO bornes VALUES (DEFAULT,%s,%s,%s,%s)", row)\n    except Exception as e:\n        print("Failed row:", row)\n        raise\n'

In [42]:
cur.executemany("INSERT INTO bornes VALUES (DEFAULT,%s,%s,%s,%s)", data_bornes_df.values.tolist())

In [43]:
data_bornes_df["code_insee_commune"]

0          None
1          None
2          None
3          None
4          None
          ...  
228491    59359
228492     None
228493     None
228494     None
228495     None
Name: code_insee_commune, Length: 228496, dtype: object

In [4]:
cur.execute("SELECT * FROM bornes LIMIT 10")
print(cur.fetchall())
cur.execute("SELECT * FROM departements LIMIT 5")
cur.fetchall()

[(1, 'ChargePoint', None, 22, None), (2, 'ChargePoint', None, 22, None), (3, 'ChargePoint', None, 11, None), (4, 'ChargePoint', None, 11, None), (5, 'ChargePoint', None, 22, None), (6, 'ChargePoint', None, 22, None), (7, 'ChargePoint', None, 22, None), (8, 'ChargePoint', None, 22, None), (9, 'ChargePoint', None, 22, None), (10, 'ChargePoint', None, 22, None)]


[('59', 'Nord', 2615635),
 ('75', 'Paris', 2103778),
 ('13', 'Bouches-du-Rhône', 2087658),
 ('6AE', "Collectivité européenne d'Alsace", 1934548),
 ('69', 'Circonscription départementale du Rhône', 1914667)]

In [47]:
#1. Le nombre de bornes pour 100 000 habitants, par département, du mieux au moins équipé.
cur.execute("SELECT d.nom,COUNT(*)::float/d.population*100000 as borne_centmille_hab FROM bornes b INNER JOIN departements d ON b.code_departement=d.code GROUP BY d.code ORDER BY borne_centmille_hab DESC")
print(cur.fetchall())

[('Vienne', 568.2426628571168), ('Calvados', 537.6063689580951), ('Aude', 427.76466621712746), ('Paris', 408.3130444371982), ('Tarn', 403.42064466770023), ('Hautes-Alpes', 393.12176319292934), ('Deux-Sèvres', 382.96613534668165), ('Manche', 382.6966445704914), ('Savoie', 362.09412216158813), ('Hautes-Pyrénées', 360.0620707243169), ('Drôme', 355.9662499737699), ('Yonne', 350.0197130620856), ("Côte-d'Or", 349.0094426957971), ('Corrèze', 326.37672012158157), ('Gironde', 324.1066363480949), ('Landes', 318.28770440759274), ('Alpes-de-Haute-Provence', 317.755007319076), ('Pyrénées-Orientales', 309.5470509504785), ('Haute-Marne', 308.91517308160707), ('Puy-de-Dôme', 291.0664862676517), ('Haute-Vienne', 288.34275270857285), ('Lozère', 287.63433831027896), ('Charente', 277.01930628921724), ('Saône-et-Loire', 276.3896712761898), ('Vosges', 275.71882837692584), ('Ain', 272.9103370310182), ('Lot-et-Garonne', 269.18303847099236), ('Allier', 269.1285276239281), ('Creuse', 267.46994209145913), ('Yvel

In [48]:
#2. Les départements les plus sous-équipés.
cur.execute("SELECT d.nom,COUNT(*) FROM bornes b INNER JOIN departements d ON b.code_departement=d.code GROUP BY d.code ORDER BY COUNT(*) ASC")
print(cur.fetchall())

[('La Réunion', 2), ('Martinique', 5), ('Guadeloupe', 141), ('Haute-Corse', 149), ('Corse-du-Sud', 198), ('Lozère', 220), ('Meuse', 232), ('Cantal', 236), ('Lot', 242), ('Gers', 262), ('Territoire de Belfort', 277), ('Ardennes', 302), ('Creuse', 309), ('Haute-Loire', 316), ('Haute-Saône', 323), ('Orne', 344), ('Jura', 397), ('Ariège', 405), ('Mayenne', 438), ('Nièvre', 472), ('Haute-Marne', 520), ('Alpes-de-Haute-Provence', 534), ('Ardèche', 551), ('Indre', 558), ('Hautes-Alpes', 564), ('Aveyron', 615), ('Tarn-et-Garonne', 648), ('Aube', 694), ('Loir-et-Cher', 705), ('Cher', 748), ('Corrèze', 786), ('Hautes-Pyrénées', 833), ('Allier', 897), ('Lot-et-Garonne', 898), ('Dordogne', 918), ('Charente', 977), ('Vosges', 985), ('Eure-et-Loir', 1030), ('Indre-et-Loire', 1043), ('Doubs', 1049), ('Somme', 1060), ('Aisne', 1067), ('Haute-Vienne', 1076), ('Sarthe', 1108), ('Yonne', 1163), ('Vendée', 1174), ('Loiret', 1185), ('Charente-Maritime', 1323), ('Landes', 1380), ('Marne', 1435), ('Deux-Sèvr

In [37]:
#3. L'opérateur dominant dans un département donné.
cur.execute("SELECT d.code, (SELECT b.nom_operateur FROM bornes b WHERE b.code_departement=d.code GROUP BY b.nom_operateur ORDER BY COUNT(*) DESC LIMIT 1) count_bornes FROM departements d") #nombre d'apaprition de l'opérateur par département
cur.fetchall()

[('59', 'Bouygues Energies & Services'),
 ('75', 'TotalEnergies Marketing France'),
 ('13', 'EVzen (SMEG DÃ©veloppement)'),
 ('6AE', None),
 ('69', 'IZIVIA'),
 ('93', 'SPIE CityNetworks'),
 ('33', 'TotalEnergies Charging Services'),
 ('92', 'IZIVIA'),
 ('44', 'Bouygues Energies & Services'),
 ('78', 'Bouygues Energies & Services'),
 ('31', 'Bouygues Energies & Services'),
 ('77', 'Bouygues Energies & Services'),
 ('62', 'Bouygues Energies & Services'),
 ('69M', None),
 ('94', 'Bouygues Energies & Services'),
 ('91', 'Bouygues Energies & Services'),
 ('38', 'EASYCHARGE'),
 ('95', 'IZIVIA'),
 ('76', 'Bouygues Energies & Services'),
 ('34', 'Bouygues Energies & Services'),
 ('67', 'Power Dot France'),
 ('06', 'IZIVIA'),
 ('35', 'Bouygues Energies et Services'),
 ('83', 'EASYCHARGE'),
 ('57', 'Power Dot France'),
 ('29', 'Bouygues Energies & Services'),
 ('974', 'Ezdrive'),
 ('74', 'EASYCHARGE'),
 ('49', 'Bouygues Energies & Services'),
 ('60', 'Bouygues Energies & Services'),
 ('56', 'Fre